# Arabic Nested NER — Cross-Domain Adaptation Pipeline
#### This notebook was executed on Kaggle Environment 2026-03-20

## 0. Installation

In [ ]:
!pip install setuptools==69.5.1
!pip install seqeval

In [ ]:
from huggingface_hub import login
try:
    login(token="YOUR_HF_TOKEN")
except:
    print("No Token")

## 1. Setup & Configuration

In [ ]:
import os
import re
import sys
import json
import copy
import random
import pickle
import logging
import itertools
import zipfile
import shutil
import glob
import natsort
from collections import Counter, namedtuple
from functools import partial
from argparse import Namespace
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer, BertModel
from seqeval.metrics import (
    classification_report, precision_score, recall_score,
    f1_score, accuracy_score,
)
from seqeval.scheme import IOB2
import matplotlib.pyplot as plt

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s\t%(name)s\t%(asctime)s\t%(message)s",
    datefmt="%a, %d %b %Y %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)


# ── Global Configuration ──────────────────────────────────────────────────
config = Namespace(
    # ── Model backbone ────────────────────────────────────────────────────
    seed=42,
    gpus=[0],
    bert_model="aubmindlab/bert-base-arabertv02",
    dropout=0.1,
    max_seq_len=512,

    # ── Supervised training (Wojood) ──────────────────────────────────────
    train_ratio=1.0,
    train_path="./Wojood/Wojood1_1_nested/train.txt",
    val_path="./Wojood/Wojood1_1_nested/val.txt",
    test_path="./Wojood/Wojood1_1_nested/test.txt",
    train_output_path="./B1",
    max_epochs=50,
    batch_size=8,
    num_workers=0,
    log_interval=10,
    lr=2e-05,
    gamma=0.95,
    overwrite=True,
    max_checkpoints=5,

    # ── Domain adaptation (Konooz unlabeled dev) ──────────────────────────
    adapt_konooz_dir="./dev-konooz",
    adapt_batch_size=4,           # smaller due to dual forward pass (src+tgt)
    adapt_epochs=5,
    adapt_lr=2e-5,
    adapt_freeze_layers=6,        # freeze bottom N BERT encoder layers
    adapt_conf_threshold=0.90,    # min teacher confidence for pseudo-labels
    adapt_st_weight=0,          # self-training loss coefficient
    adapt_domain_weight=0.31,      # domain-adversarial loss coefficient
    adapt_val_check_every=1,      # evaluate on Wojood val every N adapt epochs

    # ── SWA ───────────────────────────────────────────────────────────────
    swa_n_average=4,              # number of final checkpoints to average

    # ── Konooz domain list (shared by adaptation and inference) ───────────
    konooz_domains=[
        "Agriculture", "Art", "Economics", "Finance", "Health",
        "History", "Law", "Politics", "Science", "Sport",
    ],

    # ── Evaluation ────────────────────────────────────────────────────────
    eval_model_path="./B1",
    eval_data_path="./Wojood/Wojood1_1_nested/val.txt",
    eval_batch_size=8,

    # ── Inference — organizer-provided blinded test data ──────────────────
    infer_model_path="./B1",
    blind_test_dir="./blinded-test-data",
    output_pred_file="Laziness_pred_v1.txt",
    output_zip_file="submission.zip",
    infer_batch_size=8,

    # ── Dashboard ─────────────────────────────────────────────────────────
    results_zip_path="./B1_output",
)


def set_seed(seed: int) -> None:
    """Fix all random seeds for reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False


set_seed(config.seed)

## 2. Data Utilities

In [ ]:
ALL_TAG_TYPES = [
    "CARDINAL", "CURR", "DATE", "EVENT", "FAC", "GPE",
    "LANGUAGE", "LAW", "LOC", "MONEY", "NORP", "OCC",
    "ORDINAL", "ORG", "PERCENT", "PERS", "PRODUCT",
    "QUANTITY", "TIME", "UNIT", "WEBSITE",
]


class Token:
    """A single word token with optional gold and predicted NER tags."""

    def __init__(self, text=None, pred_tag=None, gold_tag=None):
        self.text = text
        self.gold_tag = gold_tag
        self.pred_tag = pred_tag
        self.subwords = None

    def __str__(self):
        gold = "|".join(self.gold_tag) if self.gold_tag else ""
        pred = "|".join(t["tag"] for t in self.pred_tag) if self.pred_tag else ""
        return f"{self.text}\t{gold}\t{pred}"


class Vocab:
    """Bidirectional token<->index mapping."""

    def __init__(self, counter: Counter, specials: list = []) -> None:
        self.itos = list(counter.keys()) + specials
        self.stoi = {s: i for i, s in enumerate(self.itos)}
        self.word_count = counter

    def get_itos(self) -> list:
        return self.itos

    def get_stoi(self) -> dict:
        return self.stoi

    def __len__(self) -> int:
        return len(self.itos)


def conll_to_segments(filepath: str, ratio: float = 1.0) -> list:
    """Parse a CoNLL-format file into token segments.

    Args:
        filepath: Path to a CoNLL-formatted file.
        ratio: Fraction of segments to retain (1.0 = full dataset).

    Returns:
        List of segments; each segment is a list of Token objects.
    """
    segments, current_segment = [], []
    with open(filepath, "r", encoding="utf-8") as fh:
        for line in fh.read().splitlines():
            if not line.strip():
                if current_segment:
                    segments.append(current_segment)
                    current_segment = []
            else:
                parts = line.split()
                token = Token(text=parts[0], gold_tag=parts[1:])
                current_segment.append(token)
        if current_segment:
            segments.append(current_segment)

    if ratio < 1.0:
        segments = segments[: max(1, int(len(segments) * ratio))]
    return segments

In [ ]:
def tag_vocab_by_type() -> list:
    """Build a frozen 3-label vocabulary (B-TYPE, I-TYPE, O) for each entity type.

    Returns:
        List of Vocab objects ordered to match ALL_TAG_TYPES.
    """
    per_type_vocabs = []
    for tag_type in ALL_TAG_TYPES:
        labels = [f"B-{tag_type}", f"I-{tag_type}", "O"]
        per_type_vocabs.append(Vocab(Counter({label: 1 for label in labels})))
    return per_type_vocabs


def parse_conll_files(data_paths: tuple, ratio: float = 1.0):
    """Load one or more CoNLL splits and construct vocabularies.

    Args:
        data_paths: Tuple of file paths (e.g. train, val, test).
        ratio: Fraction of segments to retain in each split.

    Returns:
        (data_splits, vocab) where data_splits is a tuple of segment lists
        and vocab is a namedtuple with fields `tags` and `tokens`.
    """
    VocabBundle = namedtuple("Vocab", ["tags", "tokens"])
    data_splits, all_token_texts = [], []

    for path in data_paths:
        split = conll_to_segments(path, ratio=ratio)
        data_splits.append(split)
        all_token_texts += [token.text for segment in split for token in segment]

    tag_vocabs = tag_vocab_by_type()

    flat_tags = ["O"] + [
        tag for tag_type in ALL_TAG_TYPES
        for tag in (f"B-{tag_type}", f"I-{tag_type}")
    ]
    flat_tag_vocab = Vocab(Counter({tag: 1 for tag in flat_tags}))
    tag_vocabs.insert(0, flat_tag_vocab)

    token_vocab = Vocab(Counter(all_token_texts), specials=["UNK"])
    return tuple(data_splits), VocabBundle(tokens=token_vocab, tags=tag_vocabs)

## 3. Dataset & Transform

In [ ]:
class NestedTagsTransform:
    """Converts a segment (list of Tokens) into model-ready tensors.

    Applies WordPiece tokenisation, aligns per-type gold tags to subwords,
    truncates to max_seq_len, and prepends [CLS] / appends [SEP].
    """

    def __init__(self, bert_model: str, vocab, max_seq_len: int = 512):
        self.tokenizer = BertTokenizer.from_pretrained(bert_model)
        self.encoder = partial(self.tokenizer.encode, max_length=max_seq_len, truncation=True)
        self.max_seq_len = max_seq_len
        self.vocab = vocab

    def __call__(self, segment: list):
        """Transform a segment into (subwords, tags, tokens, mask, valid_len)."""
        unk_token = Token(text="UNK")
        tags, tokens, subwords = [], [], []

        for token in segment:
            token.subwords = self.encoder(token.text)[1:-1] or self.encoder("[UNK]")[1:-1]
            subwords += token.subwords
            tokens += [token] + [unk_token] * (len(token.subwords) - 1)

        for vocab in self.vocab.tags[1:]:
            entity_pattern = re.compile(
                "|".join("^" + t + "$" for t in vocab.get_itos() if "-" in t)
            )
            per_subword_tags = [
                [(list(filter(entity_pattern.match, token.gold_tag)) or ["O"])[0]]
                + ["O"] * (len(token.subwords) - 1)
                for token in segment
            ]
            per_subword_tags = list(itertools.chain(*per_subword_tags))
            tags.append([vocab.get_stoi()[tag] for tag in per_subword_tags])

        if len(subwords) > self.max_seq_len - 2:
            subwords = subwords[: self.max_seq_len - 2]
            tags = [t[: self.max_seq_len - 2] for t in tags]
            tokens = tokens[: self.max_seq_len - 2]

        tokens = [unk_token] + tokens + [unk_token]
        subwords = [self.tokenizer.cls_token_id] + subwords + [self.tokenizer.sep_token_id]

        subwords = torch.LongTensor(subwords)
        tags = torch.Tensor(tags)
        o_column = torch.Tensor([vocab.get_stoi()["O"] for vocab in self.vocab.tags[1:]])
        tags = torch.column_stack((o_column, tags, o_column)).unsqueeze(0)
        mask = torch.ones_like(tags)
        return subwords, tags, tokens, mask, len(tokens)

In [ ]:
# CrossEntropyLoss receives all positions ().
TAG_PAD_VALUE = -100


class NestedTagsDataset(Dataset):
    """PyTorch Dataset wrapping a list of Token segments."""

    def __init__(
        self,
        examples=None,
        vocab=None,
        bert_model: str = "aubmindlab/bert-base-arabertv2",
        max_seq_len: int = 512,
    ):
        self.transform = NestedTagsTransform(bert_model, vocab, max_seq_len=max_seq_len)
        self.examples = examples
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, idx):
        return self.transform(self.examples[idx])

    def collate_fn(self, batch):
        """Pad a list of samples into a batch tensor."""
        subwords, tags, tokens, masks, valid_len = zip(*batch)

        subwords = pad_sequence(subwords, batch_first=True, padding_value=0)

        masks = [
            torch.nn.ConstantPad1d((0, subwords.shape[-1] - tag.shape[-1]), 0)(mask)
            for tag, mask in zip(tags, masks)
        ]
        masks = torch.cat(masks)

        padded_tags = []
        for tag in tags:
            tag = tag.squeeze(0)
            padded_rows = [
                torch.nn.ConstantPad1d(
                    (0, subwords.shape[-1] - tag[j].shape[-1]), TAG_PAD_VALUE
                )(tag[j])
                for j in range(len(self.vocab.tags[1:]))
            ]
            padded_tags.append(torch.stack(padded_rows).unsqueeze(0))
        tags = torch.cat(padded_tags)

        return subwords, tags, tokens, masks, valid_len


def get_dataloaders(
    data_splits,
    vocab,
    data_config: dict,
    batch_size: int = 32,
    num_workers: int = 0,
    shuffle: tuple = (True, False, False),
) -> list:
    """Build one DataLoader per data split.

    Args:
        data_splits: Iterable of segment lists (train, val, test, ...).
        vocab: VocabBundle namedtuple with `tags` and `tokens`.
        data_config: Dict with key `kwargs` containing `bert_model` and `max_seq_len`.
        batch_size: Samples per batch.
        num_workers: DataLoader worker processes.
        shuffle: Per-split shuffle flag tuple.

    Returns:
        List of DataLoader objects, one per split.
    """
    dataloaders = []
    for i, segments in enumerate(data_splits):
        dataset = NestedTagsDataset(
            examples=segments,
            vocab=vocab,
            bert_model=data_config["kwargs"]["bert_model"],
            max_seq_len=data_config["kwargs"]["max_seq_len"],
        )
        dataloaders.append(
            DataLoader(
                dataset=dataset,
                shuffle=shuffle[i],
                batch_size=batch_size,
                num_workers=num_workers,
                collate_fn=dataset.collate_fn,
            )
        )
    return dataloaders

## 4. Model Architecture

In [ ]:
class BaseModel(nn.Module):
    """Shared BERT backbone with dropout."""

    def __init__(
        self,
        bert_model: str = "aubmindlab/bert-base-arabertv2",
        num_labels: int = 2,
        dropout: float = 0.1,
        num_types: int = 0,
    ):
        super().__init__()
        self.bert_model = bert_model
        self.num_labels = num_labels
        self.num_types = num_types
        self.dropout = nn.Dropout(dropout)
        self.bert = BertModel.from_pretrained(bert_model)


class BertNestedTagger(BaseModel):
    """AraBERT encoder with one linear classifier head per entity type.

    Each head independently predicts (B-TYPE, I-TYPE, O) for its type,
    enabling overlapping / nested entity spans.
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.max_num_labels = max(self.num_labels)
        self.classifiers = nn.Sequential(
            *[nn.Linear(768, n) for n in self.num_labels]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run encoder + all type classifiers.

        Args:
            x: Subword token-id tensor of shape (batch, seq_len).

        Returns:
            Logit tensor of shape (batch, seq_len, num_types, max_num_labels).
        """
        hidden = self.bert(x)["last_hidden_state"]
        hidden = self.dropout(hidden)
        logits_per_type = []
        for classifier in self.classifiers:
            logits = classifier(hidden)
            logits = torch.nn.ConstantPad1d(
                (0, self.max_num_labels - logits.shape[-1]), 0
            )(logits)
            logits_per_type.append(logits)
        return torch.stack(logits_per_type).permute((1, 2, 0, 3))

## 5. Trainer

In [ ]:
def compute_nested_metrics(segments: list, tag_vocabs: list):
    """Compute nested NER metrics via seqeval.

    Args:
        segments: List of segments with gold_tag and pred_tag on each Token.
        tag_vocabs: Per-type Vocab list (matches training type ordering).

    Returns:
        SimpleNamespace with micro_f1, macro_f1, weighted_f1, precision,
        recall, and accuracy attributes.
    """
    gold_sequences, pred_sequences = [], []
    for i, vocab in enumerate(tag_vocabs):
        entity_tags = [tag for tag in vocab.get_itos() if "-" in tag]
        entity_pattern = re.compile("|".join(entity_tags))
        gold_sequences += [
            [(list(filter(entity_pattern.match, token.gold_tag)) or ["O"])[0]
             for token in segment]
            for segment in segments
        ]
        pred_sequences += [
            [token.pred_tag[i]["tag"] for token in segment]
            for segment in segments
        ]

    report = classification_report(gold_sequences, pred_sequences, scheme=IOB2, digits=4)
    print("\nClassification Report:\n", report)

    return SimpleNamespace(
        micro_f1=f1_score(gold_sequences, pred_sequences, average="micro", scheme=IOB2),
        macro_f1=f1_score(gold_sequences, pred_sequences, average="macro", scheme=IOB2),
        weighted_f1=f1_score(gold_sequences, pred_sequences, average="weighted", scheme=IOB2),
        precision=precision_score(gold_sequences, pred_sequences, scheme=IOB2),
        recall=recall_score(gold_sequences, pred_sequences, scheme=IOB2),
        accuracy=accuracy_score(gold_sequences, pred_sequences),
    )


class BaseTrainer:
    """Generic training loop with checkpointing and early stopping."""

    def __init__(
        self,
        model=None,
        max_epochs: int = 50,
        optimizer=None,
        scheduler=None,
        loss=None,
        train_dataloader=None,
        val_dataloader=None,
        test_dataloader=None,
        log_interval: int = 10,
        output_path: str = None,
        clip: float = 5,
        patience: int = 3,
        max_checkpoints: int = 1,
    ):
        self.model = model
        self.max_epochs = max_epochs
        self.train_dataloader = train_dataloader
        self.val_dataloader = val_dataloader
        self.test_dataloader = test_dataloader
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.loss = loss
        self.log_interval = log_interval
        self.output_path = output_path
        self.current_timestep = 0
        self.current_epoch = 0
        self.clip = clip
        self.patience = patience
        self.max_checkpoints = max_checkpoints
        self.history = {
            "epoch": [],
            "train_loss": [],
            "val_loss": [],
            "val_micro_f1": [],
            "val_precision": [],
            "val_recall": [],
        }

    def save(self) -> None:
        """Persist current model weights; prune old checkpoints if over limit."""
        os.makedirs(os.path.join(self.output_path, "checkpoints"), exist_ok=True)
        ckpt_path = os.path.join(
            self.output_path, "checkpoints", f"checkpoint_{self.current_epoch}.pt"
        )
        torch.save(
            {
                "model": self.model.state_dict(),
                "optimizer": self.optimizer.state_dict(),
                "epoch": self.current_epoch,
            },
            ckpt_path,
        )
        logger.info("Saved checkpoint: %s", ckpt_path)

        if self.max_checkpoints is not None:
            all_ckpts = natsort.natsorted(
                glob.glob(os.path.join(self.output_path, "checkpoints", "checkpoint_*.pt"))
            )
            for old in all_ckpts[: -self.max_checkpoints]:
                try:
                    os.remove(old)
                    logger.info("Removed old checkpoint: %s", old)
                except Exception as exc:
                    logger.warning("Could not remove checkpoint %s: %s", old, exc)

    def load(self, checkpoint_dir: str) -> None:
        """Load the latest checkpoint from checkpoint_dir."""
        ckpt_files = natsort.natsorted(
            glob.glob(f"{checkpoint_dir}/checkpoint_*.pt")
        )
        if not ckpt_files:
            raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir!r}")
        latest = ckpt_files[-1]
        logger.info("Loading checkpoint: %s", latest)
        device = None if torch.cuda.is_available() else torch.device("cpu")
        checkpoint = torch.load(latest, map_location=device, weights_only=False)
        self.model.load_state_dict(checkpoint["model"])

In [ ]:
class BertNestedTrainer(BaseTrainer):
    """Trainer for BertNestedTagger: multi-head NER loss, early stopping."""

    def train(self) -> None:
        """Run supervised training with early stopping on val micro-F1."""
        best_val_f1 = -np.inf
        num_train_batches = len(self.train_dataloader)
        tag_type_sizes = [len(v) for v in self.train_dataloader.dataset.vocab.tags[1:]]
        patience = self.patience

        for epoch_idx in range(self.max_epochs):
            self.current_epoch = epoch_idx
            epoch_train_loss = 0.0

            for batch_idx, (subwords, gold_tags, tokens, valid_len, logits) in enumerate(
                self.tag(self.train_dataloader, is_train=True), 1
            ):
                self.current_timestep += 1

                losses = [
                    self.loss(
                        logits[:, :, i, :n].view(-1, n),
                        gold_tags[:, i, :].reshape(-1).long(),
                    )
                    for i, n in enumerate(tag_type_sizes)
                ]
                sum(losses).backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.clip)
                self.optimizer.step()

                epoch_train_loss += sum(l.item() for l in losses)

                if self.current_timestep % self.log_interval == 0:
                    logger.info(
                        "Epoch %d | Batch %d/%d | Step %d | LR %.10f | Loss %.4f",
                        epoch_idx, batch_idx, num_train_batches, self.current_timestep,
                        self.optimizer.param_groups[0]["lr"],
                        sum(l.item() for l in losses),
                    )

            self.scheduler.step()
            epoch_train_loss /= num_train_batches

            logger.info("** Evaluating on validation set **")
            _, val_segments, _, val_loss = self.eval(self.val_dataloader)
            val_metrics = compute_nested_metrics(
                val_segments, self.val_dataloader.dataset.vocab.tags[1:]
            )

            self.history["epoch"].append(epoch_idx)
            self.history["train_loss"].append(epoch_train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["val_micro_f1"].append(val_metrics.micro_f1)
            self.history["val_precision"].append(val_metrics.precision)
            self.history["val_recall"].append(val_metrics.recall)

            logger.info(
                "Epoch %d | Step %d | Train Loss %.4f | Val Loss %.4f | Val F1 %.4f",
                epoch_idx, self.current_timestep, epoch_train_loss, val_loss, val_metrics.micro_f1,
            )

            if epoch_idx == 0:
                self.save()

            if val_metrics.micro_f1 > best_val_f1:
                patience = self.patience
                best_val_f1 = val_metrics.micro_f1
                logger.info("** Validation improved — evaluating test set **")
                _, test_segments, _, _ = self.eval(self.test_dataloader)
                test_metrics = compute_nested_metrics(
                    test_segments, self.test_dataloader.dataset.vocab.tags[1:]
                )
                logger.info("Test F1: %.4f", test_metrics.micro_f1)
                self.save()
            else:
                patience -= 1
                logger.info("** Patience: {patience-1} **")

            if patience == 0:
                logger.info("Early stopping triggered.")
                break

        with open(os.path.join(self.output_path, "history.json"), "w") as fh:
            json.dump(self.history, fh, indent=4)

    def tag(self, dataloader, is_train: bool = True):
        """Yield (subwords, gold_tags, tokens, valid_len, logits) per batch.

        Manages zero_grad and train/eval mode internally.
        """
        for subwords, gold_tags, tokens, mask, valid_len in dataloader:
            self.model.train(is_train)
            if torch.cuda.is_available():
                subwords = subwords.cuda()
                gold_tags = gold_tags.cuda()

            if is_train:
                self.optimizer.zero_grad()
                logits = self.model(subwords)
            else:
                with torch.no_grad():
                    logits = self.model(subwords)

            yield subwords, gold_tags, tokens, valid_len, logits

    def eval(self, dataloader):
        """Evaluate on a dataloader; returns (predictions, segments, valid_lens, loss)."""
        all_preds, all_segments, all_valid_lens = [], [], []
        tag_type_sizes = [len(v) for v in dataloader.dataset.vocab.tags[1:]]
        total_loss = 0.0

        for _, gold_tags, tokens, valid_len, logits in self.tag(dataloader, is_train=False):
            losses = [
                self.loss(
                    logits[:, :, i, :n].view(-1, n),
                    gold_tags[:, i, :].reshape(-1).long(),
                )
                for i, n in enumerate(tag_type_sizes)
            ]
            total_loss += sum(l.item() for l in losses)
            all_preds += torch.argmax(logits, dim=3)
            all_segments += tokens
            all_valid_lens += list(valid_len)

        total_loss /= len(dataloader)
        segments = self.to_segments(all_segments, all_preds, all_valid_lens, dataloader.dataset.vocab)
        return all_preds, segments, all_valid_lens, total_loss

    def infer(self, dataloader) -> list:
        """Run inference; sets pred_tag on Token objects in-place."""
        all_preds, all_segments, all_valid_lens = [], [], []
        for _, gold_tags, tokens, valid_len, logits in self.tag(dataloader, is_train=False):
            all_preds += torch.argmax(logits, dim=3)
            all_segments += tokens
            all_valid_lens += list(valid_len)
        return self.to_segments(all_segments, all_preds, all_valid_lens, dataloader.dataset.vocab)

    def to_segments(self, segments, predictions, valid_lens, vocab) -> list:
        """Assign pred_tag to each token, filtering subword continuations."""
        tokens_stoi = vocab.tokens.get_stoi()
        unk_id = tokens_stoi["UNK"]
        tagged_segments = []

        for segment, pred, valid_len in zip(segments, predictions, valid_lens):
            segment_pairs = list(filter(
                lambda t: tokens_stoi[t[0].text] != unk_id,
                zip(segment[1: valid_len - 1], pred[1: valid_len - 1]),
            ))
            for token, tag_ids in segment_pairs:
                token.pred_tag = [
                    {"tag": v.get_itos()[tag_id]}
                    for tag_id, v in zip(tag_ids.int().tolist(), vocab.tags[1:])
                ]
            tagged_segments.append([t for t, _ in segment_pairs])

        return tagged_segments

## 6. Supervised Training

In [ ]:
args = Namespace(
    output_path=config.train_output_path,
    train_path=config.train_path,
    val_path=config.val_path,
    test_path=config.test_path,
    bert_model=config.bert_model,
    gpus=config.gpus,
    log_interval=config.log_interval,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
    overwrite=config.overwrite,
    seed=config.seed,
    ratio=config.train_ratio,
    max_checkpoints=config.max_checkpoints,
    data_config={"kwargs": {"max_seq_len": config.max_seq_len, "bert_model": config.bert_model}},
    trainer_config={"kwargs": {"max_epochs": config.max_epochs}},
    network_config={"kwargs": {"dropout": config.dropout, "bert_model": config.bert_model}},
    optimizer={"fn": "torch.optim.AdamW", "kwargs": {"lr": config.lr}},
    lr_scheduler={"fn": "torch.optim.lr_scheduler.ExponentialLR", "kwargs": {"gamma": config.gamma}},
    loss={"fn": "torch.nn.CrossEntropyLoss", "kwargs": {"ignore_index":-100}},
)

os.makedirs(args.output_path, exist_ok=True)
os.makedirs(os.path.join(args.output_path, "checkpoints"), exist_ok=True)

data_splits, vocab = parse_conll_files(
    (args.train_path, args.val_path, args.test_path), ratio=args.ratio
)

with open(os.path.join(args.output_path, "tag_vocab.pkl"), "wb") as fh:
    pickle.dump(vocab.tags, fh)

args.network_config["kwargs"]["num_labels"] = [len(v) for v in vocab.tags[1:]]
with open(os.path.join(args.output_path, "args.json"), "w") as fh:
    json.dump(
        {k: (v if not isinstance(v, Namespace) else v.__dict__) for k, v in args.__dict__.items()},
        fh, indent=4,
    )

In [ ]:
train_dataloader, val_dataloader, test_dataloader = get_dataloaders(
    data_splits, vocab, args.data_config, args.batch_size, args.num_workers
)

model = BertNestedTagger(**args.network_config["kwargs"])
if torch.cuda.is_available() and len(args.gpus) > 0:
    os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in args.gpus)
    model = nn.DataParallel(model, device_ids=range(len(args.gpus))).cuda()
else:
    model = nn.DataParallel(model)

loss_fn = torch.nn.CrossEntropyLoss(**args.loss["kwargs"])
optimizer = torch.optim.AdamW(model.parameters(), lr=args.optimizer["kwargs"]["lr"])
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, **args.lr_scheduler["kwargs"])

trainer = BertNestedTrainer(
    model=model,
    max_epochs=args.trainer_config["kwargs"]["max_epochs"],
    optimizer=optimizer,
    scheduler=scheduler,
    loss=loss_fn,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    test_dataloader=test_dataloader,
    log_interval=args.log_interval,
    output_path=args.output_path,
    max_checkpoints=args.max_checkpoints,
)

In [ ]:
trainer.train()


## 6b. Domain Adaptation (Self-Training + DANN + SWA)

Uses **unlabeled** `dev-konooz` only. The blinded test set (`config.blind_test_dir`) is never loaded here.

In [ ]:
# Reload the best supervised checkpoint before adaptation.
# trainer.train() may run extra epochs after the best val point;
# the live model object may not hold the best weights.
trainer.load(os.path.join(args.output_path, "checkpoints"))
print("Best supervised checkpoint loaded for adaptation.")

In [ ]:
# read_inference_file is formally defined in Section 9.
# Re-declared here (identical) so adaptation cells can run independently.
def read_inference_file(filepath: str, num_tag_types: int = 21) -> list:
    """Load an unlabeled CoNLL file (token-per-line, blank-line delimiters)."""
    segments, current_segment = [], []
    with open(filepath, "r", encoding="utf-8") as fh:
        for line in fh.read().splitlines():
            line = line.strip()
            if not line:
                if current_segment:
                    segments.append(current_segment)
                    current_segment = []
            else:
                word = line.split()[0]
                current_segment.append(Token(text=word, gold_tag=["O"] * num_tag_types))
        if current_segment:
            segments.append(current_segment)
    return segments


adapt_segments = []
for domain in config.konooz_domains:
    fpath = os.path.join(config.adapt_konooz_dir, f"{domain}.txt")
    if not os.path.exists(fpath):
        logger.warning("Missing dev-konooz file: %s (skipped)", fpath)
        continue
    adapt_segments += read_inference_file(fpath, num_tag_types=len(vocab.tags[1:]))

assert adapt_segments, (
    f"No adaptation segments loaded from {config.adapt_konooz_dir!r}. "
    "Check the path and domain file names."
)
print(f"Loaded {len(adapt_segments)} adaptation segments across {len(config.konooz_domains)} domains.")

adapt_token_texts = [token.text for seg in adapt_segments for token in seg]
vocabs_type = namedtuple("Vocab", ["tags", "tokens"])
adapt_vocab = vocabs_type(
    tokens=Vocab(Counter(adapt_token_texts), specials=["UNK"]),
    tags=vocab.tags,
)

adapt_dataset = NestedTagsDataset(
    examples=adapt_segments,
    vocab=adapt_vocab,
    bert_model=config.bert_model,
    max_seq_len=config.max_seq_len,
)

adapt_loader = DataLoader(
    adapt_dataset,
    batch_size=config.adapt_batch_size,
    shuffle=True,
    collate_fn=adapt_dataset.collate_fn,
)
print(f"Adaptation DataLoader: {len(adapt_loader)} batches @ batch_size={config.adapt_batch_size}")

In [ ]:
bert_module    = model.module.bert        if hasattr(model, "module") else model.bert
classifiers    = model.module.classifiers if hasattr(model, "module") else model.classifiers
dropout_layer  = model.module.dropout     if hasattr(model, "module") else model.dropout
max_num_labels = model.module.max_num_labels if hasattr(model, "module") else model.max_num_labels

# Freeze embeddings + bottom N encoder layers; only top layers adapt.
n_frozen = n_trainable = 0
for name, param in bert_module.named_parameters():
    if "embeddings" in name or any(
        f"encoder.layer.{i}." in name for i in range(config.adapt_freeze_layers)
    ):
        param.requires_grad = False
        n_frozen += param.numel()
    else:
        n_trainable += param.numel()
print(f"Backbone frozen: {n_frozen:,} | trainable: {n_trainable:,}")


def forward_with_pooled(subwords: torch.Tensor):
    """Forward pass returning (logits, cls_pooled) without a second backbone call."""
    hidden = bert_module(subwords)["last_hidden_state"]
    dropped = dropout_layer(hidden)
    logits_per_type = []
    for clf in classifiers:
        logits = clf(dropped)
        logits = torch.nn.ConstantPad1d((0, max_num_labels - logits.shape[-1]), 0)(logits)
        logits_per_type.append(logits)
    logits = torch.stack(logits_per_type).permute((1, 2, 0, 3))
    cls_pooled = hidden[:, 0, :]
    return logits, cls_pooled


# Frozen snapshot of the supervised model used as pseudo-label teacher
teacher_model = copy.deepcopy(model).eval()
for param in teacher_model.parameters():
    param.requires_grad = False


class GradReverse(torch.autograd.Function):
    """Gradient Reversal Layer (Ganin et al., 2016)."""

    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambd, None


def grad_reverse(x: torch.Tensor, lambd: float = 1.0) -> torch.Tensor:
    return GradReverse.apply(x, lambd)


domain_head = nn.Sequential(
    nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, 2)
)
if torch.cuda.is_available():
    domain_head = domain_head.cuda()

print("Teacher, gradient-reversal layer, and domain classifier ready.")

In [ ]:
adapt_params = [p for p in model.parameters() if p.requires_grad] + list(domain_head.parameters())
adapt_optimizer = torch.optim.AdamW(adapt_params, lr=config.adapt_lr, weight_decay=0.01)
ner_loss_fn = torch.nn.CrossEntropyLoss(ignore_index=-100)

tag_type_sizes = [len(v) for v in vocab.tags[1:]]
adapt_iter = itertools.cycle(adapt_loader)
total_steps = config.adapt_epochs * len(train_dataloader)
global_step = 0

adapt_log = {
    "epoch": [], "ner_loss": [], "self_train_loss": [],
    "domain_loss": [], "wojood_val_loss": [], "wojood_val_micro_f1": [],
}

for epoch in range(config.adapt_epochs):
    epoch_ner_loss = epoch_st_loss = epoch_dom_loss = 0.0
    n_batches = 0

    for src_subwords, src_tags, _, _, _ in train_dataloader:
        global_step += 1
        # lambda ramp-up schedule (Ganin et al., 2016)
        lambd = 2.0 / (1.0 + np.exp(-10 * global_step / total_steps)) - 1.0

        tgt_subwords, _, _, _, _ = next(adapt_iter)
        if torch.cuda.is_available():
            src_subwords, src_tags = src_subwords.cuda(), src_tags.cuda()
            tgt_subwords = tgt_subwords.cuda()

        model.train()
        adapt_optimizer.zero_grad()

        # -- Supervised NER on labeled Wojood --
        src_logits, src_cls = forward_with_pooled(src_subwords)
        ner_loss = sum(
            ner_loss_fn(
                src_logits[:, :, i, :n].reshape(-1, n),
                src_tags[:, i, :].reshape(-1).long(),
            )
            for i, n in enumerate(tag_type_sizes)
        )

        # -- Self-training on unlabeled Konooz (confidence-gated) --
        with torch.no_grad():
            teacher_logits = teacher_model(tgt_subwords)
            teacher_probs  = torch.softmax(teacher_logits, dim=-1)
            conf, pseudo_labels = teacher_probs.max(dim=-1)

        tgt_logits, tgt_cls = forward_with_pooled(tgt_subwords)
        st_terms = []
        for i, n in enumerate(tag_type_sizes):
            high_conf_mask = conf[:, :, i] > config.adapt_conf_threshold
            if high_conf_mask.sum() == 0:
                continue
            st_terms.append(
                ner_loss_fn(
                    tgt_logits[:, :, i, :n][high_conf_mask],
                    pseudo_labels[:, :, i][high_conf_mask].clamp(max=n - 1).long(),
                )
            )
        self_train_loss = sum(st_terms) if st_terms else torch.zeros((), device=src_subwords.device)

        # -- Domain-adversarial alignment on [CLS] representations --
        domain_logits = domain_head(
            grad_reverse(torch.cat([src_cls, tgt_cls]), lambd)
        )
        domain_labels = torch.cat([
            torch.zeros(src_cls.size(0)),
            torch.ones(tgt_cls.size(0)),
        ]).long().to(domain_logits.device)
        domain_loss = nn.functional.cross_entropy(domain_logits, domain_labels)

        total_loss = ner_loss + config.adapt_st_weight * self_train_loss + config.adapt_domain_weight * domain_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(adapt_params, 5)
        adapt_optimizer.step()

        epoch_ner_loss += ner_loss.item()
        epoch_st_loss  += float(self_train_loss.detach().item())
        epoch_dom_loss += domain_loss.item()
        n_batches += 1

    epoch_ner_loss /= max(n_batches, 1)
    epoch_st_loss  /= max(n_batches, 1)
    epoch_dom_loss /= max(n_batches, 1)

    logger.info(
        "[Adapt] Epoch %d | NER %.4f | SelfTrain %.4f | Domain %.4f",
        epoch, epoch_ner_loss, epoch_st_loss, epoch_dom_loss,
    )
    adapt_log["epoch"].append(epoch)
    adapt_log["ner_loss"].append(epoch_ner_loss)
    adapt_log["self_train_loss"].append(epoch_st_loss)
    adapt_log["domain_loss"].append(epoch_dom_loss)

    if (epoch + 1) % config.adapt_val_check_every == 0:
        _, val_segments, _, val_loss = trainer.eval(val_dataloader)
        val_metrics = compute_nested_metrics(val_segments, val_dataloader.dataset.vocab.tags[1:])
        logger.info(
            "[Adapt] Epoch %d | Wojood Val Loss %.4f | Wojood Val F1 %.4f",
            epoch, val_loss, val_metrics.micro_f1,
        )
        adapt_log["wojood_val_loss"].append(val_loss)
        adapt_log["wojood_val_micro_f1"].append(val_metrics.micro_f1)
        if val_metrics.micro_f1 < 0.5:
            logger.warning(
                "Val F1 below 0.5 -- possible catastrophic forgetting. "
                "Consider lowering config.adapt_lr, adapt_st_weight, or adapt_domain_weight."
            )
            break

    trainer.current_epoch = 1000 + epoch
    trainer.save()

with open(os.path.join(args.output_path, "adaptation_history.json"), "w") as fh:
    json.dump(adapt_log, fh, indent=4)
print("Domain adaptation complete.")

In [ ]:
checkpoint_paths = natsort.natsorted(
    glob.glob(os.path.join(args.output_path, "checkpoints", "checkpoint_*.pt"))
)[-config.swa_n_average:]
print(f"Averaging {len(checkpoint_paths)} checkpoints:", [os.path.basename(p) for p in checkpoint_paths])

averaged_weights = None
for ckpt_path in checkpoint_paths:
    checkpoint_weights = torch.load(ckpt_path, map_location="cpu", weights_only=False)["model"]
    if averaged_weights is None:
        averaged_weights = {k: v.clone().float() for k, v in checkpoint_weights.items()}
    else:
        for k in averaged_weights:
            averaged_weights[k] += checkpoint_weights[k].float()

for k in averaged_weights:
    averaged_weights[k] /= len(checkpoint_paths)

# BERT uses LayerNorm (not BatchNorm), so uniform weight averaging is safe --
# no running-statistics recomputation pass is needed.
swa_ckpt_path = os.path.join(args.output_path, "checkpoints", "checkpoint_9999.pt")
torch.save({"model": averaged_weights, "optimizer": adapt_optimizer.state_dict(), "epoch": 9999}, swa_ckpt_path)
print(f"SWA checkpoint saved: {swa_ckpt_path}")

remaining = natsort.natsorted(
    glob.glob(os.path.join(args.output_path, "checkpoints", "checkpoint_*.pt"))
)
print("Checkpoints on disk:", [os.path.basename(p) for p in remaining])
print("BaseTrainer.load() picks the last natsorted checkpoint -> checkpoint_9999.pt")

## 7. Training Dashboard

In [ ]:
output_dir   = config.train_output_path
history_file = os.path.join(output_dir, "history.json")

if os.path.exists(history_file):
    with open(history_file) as fh:
        history = json.load(fh)
    epochs = history["epoch"]

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["train_loss"], label="Train Loss", color="#1f77b4", marker="o", linewidth=2)
    plt.plot(epochs, history["val_loss"],   label="Val Loss",   color="#ff7f0e", marker="o", linewidth=2)
    plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold")
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend(); plt.grid(True, linestyle="--", alpha=0.6)
    plt.savefig(os.path.join(output_dir, "loss_curves.png"), dpi=150, bbox_inches="tight")
    plt.close()

    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history["val_micro_f1"],  label="Micro F1",  color="#2ca02c", marker="s", linewidth=2)
    plt.plot(epochs, history["val_precision"], label="Precision", color="#d62728", marker="x", linewidth=1.5, linestyle="--")
    plt.plot(epochs, history["val_recall"],    label="Recall",    color="#9467bd", marker="^", linewidth=1.5, linestyle="--")
    plt.title("Validation Metrics", fontsize=14, fontweight="bold")
    plt.xlabel("Epoch"); plt.ylabel("Score")
    plt.legend(); plt.grid(True, linestyle="--", alpha=0.6)
    plt.savefig(os.path.join(output_dir, "metrics_curves.png"), dpi=150, bbox_inches="tight")
    plt.close()

    print(f"Loss and metrics plots saved to {output_dir}.")

In [ ]:
if "trainer" in globals() and hasattr(trainer, "eval") and "val_dataloader" in globals():
    print("Evaluating best checkpoint for per-class F1 chart...")
    try:
        trainer.load(os.path.join(output_dir, "checkpoints"))
        _, eval_segments, _, _ = trainer.eval(val_dataloader)

        gold_sequences, pred_sequences = [], []
        for i, tag_vocab in enumerate(val_dataloader.dataset.vocab.tags[1:]):
            entity_tags = [t for t in tag_vocab.get_itos() if "-" in t]
            pattern = re.compile("|".join(entity_tags))
            gold_sequences += [
                [(list(filter(pattern.match, token.gold_tag)) or ["O"])[0] for token in seg]
                for seg in eval_segments
            ]
            pred_sequences += [[token.pred_tag[i]["tag"] for token in seg] for seg in eval_segments]

        report_str = classification_report(gold_sequences, pred_sequences, scheme=IOB2, digits=4)
        with open(os.path.join(output_dir, "final_classification_report.txt"), "w", encoding="utf-8") as fh:
            fh.write(report_str)

        try:
            report_dict = classification_report(gold_sequences, pred_sequences, scheme=IOB2, output_dict=True)
        except Exception:
            report_dict = {}
            for line in report_str.split("\n")[2:-4]:
                parts = line.strip().split()
                if len(parts) >= 5:
                    report_dict[parts[0]] = {"f1-score": float(parts[3])}

        class_names = [k for k in report_dict if k not in ("micro avg", "macro avg", "weighted avg")]
        if class_names:
            f1_scores = [report_dict[c]["f1-score"] for c in class_names]
            sorted_classes, sorted_f1s = zip(*sorted(zip(class_names, f1_scores), key=lambda x: x[1]))

            plt.figure(figsize=(10, 8))
            plt.barh(sorted_classes, sorted_f1s, color="#17becf", edgecolor="black", height=0.6)
            plt.title("F1-Score per Entity Type (Validation)", fontsize=14, fontweight="bold")
            plt.xlabel("F1-Score"); plt.ylabel("Entity Type")
            plt.xlim(0, 1.05); plt.grid(axis="x", linestyle="--", alpha=0.6)
            for i, val in enumerate(sorted_f1s):
                plt.text(val + 0.01, i, f"{val:.3f}", va="center", fontsize=9, fontweight="bold")
            plt.savefig(os.path.join(output_dir, "per_class_f1.png"), dpi=150, bbox_inches="tight")
            plt.close()
            print(f"Per-class F1 chart saved to {output_dir}.")
    except Exception as exc:
        print(f"Skipped checkpoint evaluation: {exc}")

In [ ]:
if os.path.exists(history_file):
    best_epoch_idx = int(np.argmin(history["val_loss"]))
    summary_md = (
        f"# Wojood Nested NER — Training Summary\n\n"
        f"## Configuration\n"
        f"- **BERT model**: `{config.bert_model}`\n"
        f"- **Dropout**: `{config.dropout}`\n"
        f"- **Batch size**: `{config.batch_size}`\n"
        f"- **Seed**: `{config.seed}`\n"
        f"- **Train ratio**: `{config.train_ratio}`\n\n"
        f"## Best Epoch (Epoch {best_epoch_idx})\n"
        f"- Train Loss: `{history['train_loss'][best_epoch_idx]:.6f}`\n"
        f"- Val Loss: `{history['val_loss'][best_epoch_idx]:.6f}`\n"
        f"- Val Micro F1: `{history['val_micro_f1'][best_epoch_idx]:.6f}`\n"
        f"- Val Precision: `{history['val_precision'][best_epoch_idx]:.6f}`\n"
        f"- Val Recall: `{history['val_recall'][best_epoch_idx]:.6f}`\n"
    )
    with open(os.path.join(output_dir, "results_summary.md"), "w", encoding="utf-8") as fh:
        fh.write(summary_md)

checkpoint_dir = os.path.join(output_dir, "checkpoints")
ckpt_files = natsort.natsorted(glob.glob(os.path.join(checkpoint_dir, "checkpoint_*.pt")))
if ckpt_files:
    best_ckpt_dest = os.path.join(output_dir, "best_model.pt")
    shutil.copy(ckpt_files[-1], best_ckpt_dest)
    print(f"Best checkpoint copied to: {best_ckpt_dest}")

# if os.path.exists(output_dir):
#     print(f"Zipping: {output_dir} -> {config.results_zip_path}.zip")
#     shutil.make_archive(config.results_zip_path, "zip", output_dir)
#     print("Zip complete.")

In [ ]:
# ── Section 7.5: Extended Reporting Exports ────────────────────────────────────
# Reads from existing on-disk JSON/TXT files -- no re-training required.
import csv

output_dir  = config.train_output_path
history_file = os.path.join(output_dir, "history.json")
adapt_file   = os.path.join(output_dir, "adaptation_history.json")
report_file  = os.path.join(output_dir, "final_classification_report.txt")

# ── 1. Precision vs. Recall across training epochs ─────────────────────────
if os.path.exists(history_file):
    with open(history_file) as fh:
        history = json.load(fh)
    epochs = history["epoch"]
    best_f1_idx = int(np.argmax(history["val_micro_f1"]))

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs, history["val_precision"], label="Precision", color="#d62728", marker="x", linewidth=1.8)
    ax.plot(epochs, history["val_recall"],    label="Recall",    color="#9467bd", marker="^", linewidth=1.8)
    ax.axvline(epochs[best_f1_idx], color="gray", linestyle="--", alpha=0.7,
               label=f"Best F1 epoch ({epochs[best_f1_idx]})")
    ax.set_title("Validation Precision & Recall per Epoch", fontsize=14, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Score")
    ax.legend(); ax.grid(True, linestyle="--", alpha=0.6)
    fig.savefig(os.path.join(output_dir, "precision_recall_curves.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: precision_recall_curves.png")

# ── 2. Dual-axis: Val Loss vs Val F1 ───────────────────────────────────────
if os.path.exists(history_file):
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()
    ax1.plot(epochs, history["val_loss"],     color="#d62728", marker="o", linewidth=2, label="Val Loss")
    ax2.plot(epochs, history["val_micro_f1"], color="#2ca02c", marker="s", linewidth=2,
             label="Val Micro F1", linestyle="--")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Validation Loss",    color="#d62728")
    ax2.set_ylabel("Validation Micro F1", color="#2ca02c")
    ax1.tick_params(axis="y", labelcolor="#d62728")
    ax2.tick_params(axis="y", labelcolor="#2ca02c")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")
    fig.suptitle("Val Loss vs. Val F1 (Dual Axis)", fontsize=14, fontweight="bold")
    ax1.grid(True, linestyle="--", alpha=0.5)
    fig.savefig(os.path.join(output_dir, "f1_vs_loss.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: f1_vs_loss.png")

# ── 3. Per-class Precision & Recall grouped bar chart ────────────────────────
classes, precisions, recalls, f1s, supports = [], [], [], [], []
if os.path.exists(report_file):
    with open(report_file, encoding="utf-8") as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) == 5 and parts[0] not in ("micro", "macro", "weighted"):
                try:
                    classes.append(parts[0])
                    precisions.append(float(parts[1]))
                    recalls.append(float(parts[2]))
                    f1s.append(float(parts[3]))
                    supports.append(int(parts[4]))
                except ValueError:
                    pass

    if classes:
        sorted_data = sorted(zip(f1s, classes, precisions, recalls), reverse=False)
        f1s_s, classes_s, prec_s, rec_s = zip(*sorted_data)
        x = np.arange(len(classes_s))
        width = 0.35

        fig, ax = plt.subplots(figsize=(12, 9))
        ax.barh(x - width / 2, prec_s, width, label="Precision", color="#1f77b4", edgecolor="black")
        ax.barh(x + width / 2, rec_s,  width, label="Recall",    color="#ff7f0e", edgecolor="black")
        ax.set_yticks(x); ax.set_yticklabels(classes_s)
        ax.set_xlabel("Score")
        ax.set_title("Precision & Recall per Entity Type (Validation)", fontsize=14, fontweight="bold")
        ax.legend(); ax.grid(axis="x", linestyle="--", alpha=0.6)
        ax.set_xlim(0, 1.1)
        fig.savefig(os.path.join(output_dir, "per_class_precision_recall.png"), dpi=150, bbox_inches="tight")
        plt.close(fig)
        print("Saved: per_class_precision_recall.png")

# ── 4. Adaptation loss components ────────────────────────────────────────────
adapt_log = {}
if os.path.exists(adapt_file):
    with open(adapt_file) as fh:
        adapt_log = json.load(fh)
    adapt_epochs = adapt_log["epoch"]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(adapt_epochs, adapt_log["ner_loss"],        label="NER Loss",
            marker="o", linewidth=2, color="#1f77b4")
    ax.plot(adapt_epochs, adapt_log["self_train_loss"], label="Self-Train Loss",
            marker="s", linewidth=2, color="#2ca02c")
    ax.plot(adapt_epochs, adapt_log["domain_loss"],     label="Domain Adv. Loss",
            marker="^", linewidth=2, color="#d62728")
    ax.set_title("Domain Adaptation Loss Components per Epoch", fontsize=14, fontweight="bold")
    ax.set_xlabel("Adaptation Epoch"); ax.set_ylabel("Loss")
    ax.legend(); ax.grid(True, linestyle="--", alpha=0.6)
    fig.savefig(os.path.join(output_dir, "adaptation_loss_curves.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: adaptation_loss_curves.png")

# ── 5. Wojood Val F1 across adaptation epochs ──────────────────────────────
if adapt_log.get("wojood_val_micro_f1"):
    val_f1_epochs = adapt_log["epoch"][:len(adapt_log["wojood_val_micro_f1"])]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(val_f1_epochs, adapt_log["wojood_val_micro_f1"],
            label="Wojood Val Micro F1", marker="D", linewidth=2, color="#17becf")
    ax.set_title("Wojood Val Micro F1 During Domain Adaptation", fontsize=14, fontweight="bold")
    ax.set_xlabel("Adaptation Epoch"); ax.set_ylabel("Micro F1")
    ax.set_ylim(0.85, 1.0)
    ax.legend(); ax.grid(True, linestyle="--", alpha=0.6)
    fig.savefig(os.path.join(output_dir, "adaptation_val_f1.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: adaptation_val_f1.png")

# ── 6. metrics_table.csv ──────────────────────────────────────────────────
if classes:
    csv_path = os.path.join(output_dir, "metrics_table.csv")
    sorted_csv = sorted(zip(f1s, classes, precisions, recalls, supports), reverse=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as csvf:
        writer = csv.writer(csvf)
        writer.writerow(["Entity", "Precision", "Recall", "F1", "Support"])
        for f, c, p, r, s in sorted_csv:
            writer.writerow([c, f"{p:.4f}", f"{r:.4f}", f"{f:.4f}", s])
    print("Saved: metrics_table.csv")

# ── 7. metrics_table.png (rendered table image) ────────────────────────────
if classes:
    sorted_table = sorted(zip(f1s, classes, precisions, recalls, supports), reverse=True)
    col_labels = ["Entity", "Precision", "Recall", "F1-Score", "Support"]
    table_data = [[c, f"{p:.4f}", f"{r:.4f}", f"{f:.4f}", str(s)]
                  for f, c, p, r, s in sorted_table]

    fig, ax = plt.subplots(figsize=(9, len(table_data) * 0.42 + 1.2))
    ax.axis("off")
    tbl = ax.table(cellText=table_data, colLabels=col_labels, loc="center", cellLoc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9.5)
    tbl.scale(1, 1.4)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor("#2c3e50")
            cell.set_text_props(color="white", fontweight="bold")
        elif row % 2 == 0:
            cell.set_facecolor("#ecf0f1")
    ax.set_title("Per-Entity Metrics (Validation Set)", fontsize=13, fontweight="bold", pad=12)
    fig.savefig(os.path.join(output_dir, "metrics_table.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("Saved: metrics_table.png")

# ── 8. training_summary_extended.md ────────────────────────────────────────
if os.path.exists(history_file) and classes:
    best_f1_epoch = epochs[int(np.argmax(history["val_micro_f1"]))]
    best_f1_val   = max(history["val_micro_f1"])
    best_adapt_f1 = max(adapt_log.get("wojood_val_micro_f1", [0.0]))

    entity_rows = "\n".join(
        f"| {c} | {p:.4f} | {r:.4f} | {f:.4f} | {s} |"
        for f, c, p, r, s in sorted(zip(f1s, classes, precisions, recalls, supports), reverse=True)
    )
    extended_md = (
        f"# Wojood Nested NER \u2014 Extended Training Report\n\n"
        f"## Supervised Training\n"
        f"- **Model**: `{config.bert_model}`\n"
        f"- **Total epochs run**: {len(epochs)}\n"
        f"- **Best epoch (highest val F1)**: Epoch {best_f1_epoch}\n"
        f"- **Best val Micro F1**: `{best_f1_val:.6f}`\n"
        f"- **Best val Precision**: `{max(history['val_precision']):.6f}`\n"
        f"- **Best val Recall**: `{max(history['val_recall']):.6f}`\n\n"
        f"## Domain Adaptation\n"
        f"- **Adaptation epochs**: {len(adapt_log.get('epoch', []))}\n"
        f"- **Best Wojood val F1 post-adaptation**: `{best_adapt_f1:.6f}`\n\n"
        f"## Per-Entity Results (Sorted by F1, Validation Set)\n\n"
        f"| Entity | Precision | Recall | F1-Score | Support |\n"
        f"|--------|-----------|--------|----------|---------||\n"
        f"{entity_rows}\n"
    )
    with open(os.path.join(output_dir, "training_summary_extended.md"), "w", encoding="utf-8") as fh:
        fh.write(extended_md)
    print("Saved: training_summary_extended.md")

print("\nAll extended reporting exports complete.")


## 8. Evaluation

In [ ]:
eval_model_path = config.eval_model_path
eval_data_path  = config.eval_data_path

with open(os.path.join(eval_model_path, "tag_vocab.pkl"), "rb") as fh:
    eval_tag_vocab = pickle.load(fh)

eval_config = Namespace()
with open(os.path.join(eval_model_path, "args.json")) as fh:
    eval_config.__dict__ = json.load(fh)

eval_data_splits, eval_raw_vocab = parse_conll_files((eval_data_path,), ratio=1.0)
vocabs_type = namedtuple("Vocab", ["tags", "tokens"])
eval_vocab = vocabs_type(tokens=eval_raw_vocab.tokens, tags=eval_tag_vocab)

eval_dataloader = get_dataloaders(
    eval_data_splits, eval_vocab, eval_config.data_config,
    batch_size=config.eval_batch_size, shuffle=(False,),
)[0]

eval_model = BertNestedTagger(**eval_config.network_config["kwargs"])
eval_model = (
    nn.DataParallel(eval_model).cuda()
    if torch.cuda.is_available()
    else nn.DataParallel(eval_model)
)
eval_loss_fn = torch.nn.CrossEntropyLoss(**eval_config.loss["kwargs"])

eval_trainer = BertNestedTrainer(model=eval_model, loss=eval_loss_fn, output_path=eval_model_path)
eval_trainer.load(os.path.join(eval_model_path, "checkpoints"))

In [ ]:
_, eval_segments, _, eval_loss = eval_trainer.eval(eval_dataloader)
eval_metrics = compute_nested_metrics(eval_segments, eval_vocab.tags[1:])
print(f"Validation Loss : {eval_loss:.4f}")
print(f"Micro F1        : {eval_metrics.micro_f1:.4f}")

## 9. Inference & Submission

In [ ]:
def read_inference_file(filepath: str, num_tag_types: int = 21) -> list:
    """Load an unlabeled CoNLL file (token per line, blank-line sentence breaks).

    Args:
        filepath: Path to the file.
        num_tag_types: Number of entity types (fills gold_tag with O placeholders).

    Returns:
        List of segments; each segment is a list of Token objects.
    """
    segments, current_segment = [], []
    with open(filepath, "r", encoding="utf-8") as fh:
        for line in fh.read().splitlines():
            line = line.strip()
            if not line:
                if current_segment:
                    segments.append(current_segment)
                    current_segment = []
            else:
                word = line.split()[0]
                current_segment.append(Token(text=word, gold_tag=["O"] * num_tag_types))
        if current_segment:
            segments.append(current_segment)
    return segments


def save_inference_to_txt(segments: list, output_path: str) -> None:
    """Write predicted NER tags to a CoNLL-style text file.

    Args:
        segments: List of segments with pred_tag set on each Token.
        output_path: Destination file path.
    """
    with open(output_path, "w", encoding="utf-8") as fh:
        for i, segment in enumerate(segments):
            for token in segment:
                tags = " ".join(t["tag"] for t in token.pred_tag)
                fh.write(f"{token.text} {tags}\n")
            if i < len(segments) - 1:
                fh.write("\n")

In [ ]:
infer_model_path = config.infer_model_path
blind_test_dir   = config.blind_test_dir
output_pred_file = config.output_pred_file
output_zip_file  = config.output_zip_file

with open(os.path.join(infer_model_path, "tag_vocab.pkl"), "rb") as fh:
    infer_tag_vocab = pickle.load(fh)

infer_config = Namespace()
with open(os.path.join(infer_model_path, "args.json")) as fh:
    infer_config.__dict__ = json.load(fh)

num_tag_types = len(infer_tag_vocab[1:])
vocabs_type   = namedtuple("Vocab", ["tags", "tokens"])

infer_model = BertNestedTagger(**infer_config.network_config["kwargs"])
infer_model = (
    nn.DataParallel(infer_model).cuda()
    if torch.cuda.is_available()
    else nn.DataParallel(infer_model)
)
infer_trainer = BertNestedTrainer(model=infer_model, output_path=infer_model_path)
infer_trainer.load(os.path.join(infer_model_path, "checkpoints"))

In [ ]:
all_predicted_segments = []
empty_pred = [{"tag": "O"}] * num_tag_types

for domain in config.konooz_domains:
    domain_file = os.path.join(blind_test_dir, f"{domain}.txt")
    print(f"Inferring: {domain}")

    domain_segments = read_inference_file(domain_file, num_tag_types=num_tag_types)
    domain_token_texts = [token.text for seg in domain_segments for token in seg]
    domain_vocab = vocabs_type(
        tokens=Vocab(Counter(domain_token_texts), specials=["UNK"]),
        tags=infer_tag_vocab,
    )
    domain_loader = get_dataloaders(
        [domain_segments], domain_vocab, infer_config.data_config,
        batch_size=config.infer_batch_size, shuffle=(False,),
    )[0]

    # infer() sets pred_tag in-place. Tokens beyond the 512-subword window
    # remain with pred_tag=None and are filled with 'O' below.
    infer_trainer.infer(domain_loader)

    for segment in domain_segments:
        for token in segment:
            if token.pred_tag is None:
                token.pred_tag = empty_pred[:]
    all_predicted_segments.extend(domain_segments)

save_inference_to_txt(all_predicted_segments, output_pred_file)
print(f"Predictions written to: {output_pred_file}")

In [ ]:
with zipfile.ZipFile(output_zip_file, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(output_pred_file, os.path.basename(output_pred_file))
print(f"Submission zip: {output_zip_file}")